# scikit-learn — Predict 4h direction, properly

## Project goal

Build a complete supervised-learning pipeline for predicting the sign of BTC's next 4h log return: features → chronological split → fit three model candidates → CV-score them → pick the winner → calibrate it → choose a threshold → evaluate on test. Every step uses sklearn correctly.


## Why this exercises the cheatsheet

This forces every section of the sklearn cheatsheet: pipelines (with preprocessing), TimeSeriesSplit, cross_val_score with the right scoring string, predict_proba, calibration via FrozenEstimator, permutation importance, threshold selection, and the metrics module.


## Sub-task 1: Features and target

Build features (lag returns at 1h/4h/24h, rolling vol, hour-of-day, z-scored volume) and the target = `(log(close).shift(-4) > log(close)).astype(int)`. All features past-only. Drop NaN rows.

**Patterns this forces:**

- feature engineering with strict `shift()` first
- vectorised target construction
- `.dropna()` after feature build


In [ ]:
# Your answer here

## Sub-task 2: Chronological train/val/test split

Split chronologically into 70/15/15. Apply a `horizon=4` purge at each boundary so the last 4 rows of train/val don't leak target info into the next slice. Assert that the splits are temporally disjoint.

**Patterns this forces:**

- manual chronological slicing (NOT `train_test_split(shuffle=True)`)
- purge: drop last `horizon` rows of train at the train/val boundary
- explicit `assert train.index.max() < val.index.min()` etc


In [ ]:
# Your answer here

## Sub-task 3: Three pipeline-wrapped candidates

Define three models in scaling pipelines using `make_pipeline`: (a) LogisticRegression, (b) RandomForestClassifier, (c) GradientBoostingClassifier (or LightGBM). Use `class_weight='balanced'` where it applies. Use `clone()` so each fold gets a fresh instance.

**Patterns this forces:**

- `make_pipeline(StandardScaler(), LogisticRegression())`
- `from sklearn.base import clone`
- `class_weight='balanced'`


In [ ]:
# Your answer here

## Sub-task 4: CV-score the three candidates

Use `TimeSeriesSplit(5)` inside the training set. Score each candidate via `cross_validate` with `scoring=['roc_auc', 'neg_log_loss', 'accuracy']`. Build a results DataFrame with mean ± std per metric per model. Pick the winner.

**Patterns this forces:**

- `cross_validate(..., cv=TimeSeriesSplit(5), scoring=[...])`
- result dict → DataFrame
- remember `neg_` prefix on log_loss


In [ ]:
# Your answer here

## Sub-task 5: Calibrate the winner

Wrap the winner's prefit instance in `FrozenEstimator` and calibrate via `CalibratedClassifierCV(method='isotonic', cv=5)` on the val set. Compare raw vs calibrated log_loss on val.

**Patterns this forces:**

- `from sklearn.frozen import FrozenEstimator`
- `CalibratedClassifierCV(FrozenEstimator(model), method='isotonic', cv=5)`
- log_loss comparison


In [ ]:
# Your answer here

## Sub-task 6: Permutation importance

Run `permutation_importance` on val with `n_repeats=5, scoring='neg_log_loss'`. Sort and print top-5. Comment in the cell: do the top features make domain sense?

**Patterns this forces:**

- `permutation_importance(..., n_repeats=5)`
- `pd.Series(..., index=X.columns).sort_values()`
- qualitative interpretation in a comment


In [ ]:
# Your answer here

## Sub-task 7: Pick threshold on val, evaluate on test

Sweep thresholds on val to maximise F1. **Apply that threshold to test** (do not re-tune). Report final test (accuracy, F1, AUC, log_loss). Also compute a 95% bootstrap CI on test AUC.

**Patterns this forces:**

- `np.linspace(0.30, 0.70, 81)` threshold sweep
- `np.argmax([f1_score(y_val, ...) for t in ts])`
- bootstrap CI via `rng.integers(0, n, n)` resample


In [ ]:
# Your answer here

## What success looks like

- A single results table of the three candidates with CV mean/std per metric.
- A second table of (raw vs calibrated) log-loss on val.
- A final test-set scoreboard with point estimates AND a bootstrap CI on AUC.
- A clear winner with justification. The threshold was tuned on val and evaluated on test exactly once.
- Zero `KFold(shuffle=True)` anywhere in the notebook.
